Trim conversation history to fit a strict token budget. Use a real tokenizer, because message count is a poor proxy for context window usage.

Think of packing a suitcase with a strict weight limit. You don't count the number of items. You weigh each one and remove the oldest items until everything fits. Token Buffer Memory works the same way. It measures conversation history by tokens (not message count) and removes the oldest messages until the total fits under a budget.

Sliding window memory (technique 02) drops messages based on a fixed count. But one message with a code block might cost thousands of tokens, while a short "yes" costs one. Token Buffer Memory replaces that blunt count with precise measurement. A tokenizer is a tool that splits text into tokens (the word-pieces a model processes internally). Integrate a library like tiktoken. Now the system knows exactly how many tokens the history consumes before it reaches the model.

This precision matters most in production. An oversized prompt either truncates silently or throws a hard error. An undersized prompt wastes expensive context window space. Token Buffer Memory eliminates both failure modes. It sets a hard ceiling (max_token_limit) and evicts the oldest messages when the total exceeds it.

The technique is deliberately not fancy: no summarization, no embeddings (vector representations of text), no retrieval. That simplicity is its strength. It's deterministic, fast, and straightforward to reason about. When you need more sophisticated retention, layer this with summary or retrieval-augmented memory. Token Buffer Memory remains the foundational guardrail that keeps the prompt within bounds.

By the end you'll understand:

How tokenizers measure real context usage and why token counts beat message counting.

How to build a token-aware memory from scratch with tiktoken and the OpenAI SDK.

When token budgets shine and when you need something more.

Mermaid source flowchart LR

    NewMsg["New Message"] --> Tokenizer["Tokenizer\n(tiktoken)"]

    Tokenizer --> Counter["Token Counter\n(sum all messages)"]

    Counter --> BudgetCheck{"Total > \nmax_token_limit?"}

    BudgetCheck -- No --> TrimmedHistory["Trimmed History"]

    BudgetCheck -- Yes --> Evict["Evict Oldest\nMessage"]


    Evict --> Counter

    TrimmedHistory --> LLM["LLM"]

    LLM --> Response["Response"]

    Response --> NewMsg

## What Is Token Buffer Memory?

### The core idea in one sentence
Maintain a message buffer that **never exceeds a fixed token budget** — when adding a new message would breach the limit, drop the oldest messages (one at a time) until there is room.

---

### The mental model — a fixed-length ticket tape

Imagine a physical ticket tape of exactly 1,000 characters. New messages are printed on the right. When the tape is full and a new message arrives, characters are torn off the left until the new message fits. The tape always has exactly the same maximum length. The printer never slows down waiting for a compression job.

```
Token budget = 500 tokens

After Turn 3:  [T1-user 60t][T1-ai 80t][T2-user 55t][T2-ai 90t][T3-user 70t]  = 355t ✅
New message arrives (T3-ai = 120t) → total would be 475t ✅ fits, no eviction

Turn 4 arrives (T4-user = 65t) → total would be 540t ❌ over budget
  → Drop T1-user (60t) → 480t still over
  → Drop T1-ai   (80t) → 400t now fits ✅
  → Add T4-user  (65t) → 465t final
```

---

### How it differs from Technique 02 (Sliding Window)

Sliding Window drops by **turn count** — always drops N complete turns at once.
Token Buffer drops by **individual messages** until the budget fits — may drop half a turn if needed.

| | Sliding Window (02) | Token Buffer (05) |
|---|---|---|
| Control unit | Turns | Tokens |
| Eviction granularity | Whole turns (2 messages) | One message at a time |
| Budget precision | Approximate | Exact |
| Turn pair integrity | Always preserved | May be broken |
| Compression calls | None | None |
| Latency impact | None | None |

---

### How it differs from Technique 04 (Summary Buffer)

Summary Buffer compresses evicted messages into a summary — information is preserved in compressed form.
Token Buffer simply drops evicted messages — they are gone from the active context entirely.

| | Summary Buffer (04) | Token Buffer (05) |
|---|---|---|
| On overflow | Compress into summary | Drop — no recovery |
| Information loss | Partial (lossy compression) | Total (hard eviction) |
| Extra LLM calls | Yes — summarisation | None |
| Latency spike on overflow | Yes | None |
| Cost certainty | Budget + summary overhead | Exact budget only |

---

### Point 1 — Token Buffer is the simplest, most predictable short-term memory

No compression calls. No latency spikes. No external dependencies. The token budget is a hard ceiling and the cost model is:

```
Input tokens per call = system_prompt_tokens + min(history_tokens, max_buffer_tokens)
```

That formula never changes regardless of session length. You can calculate the worst-case API cost per call before writing a single line of application code.

---

### Point 2 — It is the right choice when latency is the top constraint

Summary Buffer (Technique 04) is better for memory quality — but it adds latency every time overflow fires because it makes an extra LLM API call for summarisation. In latency-sensitive applications (voice agents, real-time assistants, high-frequency trading advisors) that extra call is unacceptable. Token Buffer has zero overhead on overflow — eviction is pure in-memory list manipulation, completing in microseconds.

---

### Point 3 — Turn pair integrity is a production concern

Because Token Buffer evicts one message at a time, it can break turn pairs. If a user message is evicted but its corresponding assistant response is not, the context contains an orphaned assistant reply with no preceding question. This confuses the model.

The production fix: always evict in pairs — when you evict one message, evict the corresponding partner too. We implement this with an `evict_in_pairs` flag (default True). This adds a tiny extra cost in edge cases but prevents context incoherence.

---

### Point 4 — This is LangChain's `ConversationTokenBufferMemory`

LangChain implements this exact pattern as `ConversationTokenBufferMemory`. Building it from scratch here gives you the internals LangChain abstracts away — so you can tune, debug, and extend it for your production domain rather than accepting defaults.

---

### Point 5 — Token Buffer + external retrieval = production-ready

On its own, Token Buffer loses information on eviction. Paired with a retrieval layer (Technique 06 — Vector Store), evicted messages are not lost — they are stored in a searchable archive. When the current query is relevant to an evicted turn, the retrieval layer surfaces it. This pairing — Token Buffer for recency, Vector Store for history — is one of the most common patterns in production agent systems.

---

## FinCoach context

Token Buffer is the right choice for FinCoach in two scenarios:
1. **High-frequency sessions** — users checking markets or asking quick questions. Low latency matters more than perfect recall.
2. **As the active-window layer in a larger system** — where a vector store or entity store handles long-term memory, and Token Buffer simply manages the current conversation window.

It is not the right choice for long, information-dense advisory sessions where salary, goals, and risk profile must survive across many turns without a retrieval layer.

---

## Trade-offs

| | |
|---|---|
| ✅ Exact, predictable token cost every call | ❌ Evicted messages are gone — no recovery |
| ✅ Zero overhead on overflow — no extra API calls | ❌ May lose critical early facts |
| ✅ Zero latency spike | ❌ Turn pair integrity needs explicit handling |
| ✅ Simplest implementation of the five techniques | ❌ Not sufficient alone for information-dense sessions |

## Production Verdict

> **The right choice when latency and cost certainty outweigh memory completeness.**
>
> Token Buffer is the operational default for latency-sensitive systems and the active-window component of larger memory architectures. Use it when you have a retrieval layer that catches evicted facts, or when short focused conversations are the norm and early-turn recall is not required.

---

In [1]:
# Install required packages.
# 'openai'   — OpenAI SDK for GPT-4o API calls.
# 'tiktoken' — GPT-4o's exact tokeniser for precise budget enforcement.
!pip install openai tiktoken --quiet

In [2]:
# --- Standard library ---
import json                              # Session persistence.
import os                                # Environment variables.
import time                              # Rate-limit delays.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ safe.
from typing import List, Dict, Optional, Tuple  # Type hints.
from dataclasses import dataclass, field, asdict  # Data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.

In [3]:
# --- API Key Setup ---
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    # Colab Secrets vault — recommended: key never appears in notebook output.
    print("Key loaded from Colab Secrets.")
except Exception:
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    # Environment variable fallback for local Jupyter / VS Code.
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets."

client = openai.OpenAI(api_key=OPENAI_API_KEY)
# Single client instance — reused for all API calls in this notebook.
# Token Buffer makes NO extra API calls (no summarisation) — this client
# is used ONLY for the main FinCoach conversation calls.

MODEL = "gpt-4o"
# gpt-4o — the main conversation model.
# Note: unlike Techniques 03 and 04, there is NO SUMMARY_MODEL here.
# Token Buffer makes no summarisation calls — no second model needed.

TOKENISER = tiktoken.get_encoding("o200k_base")
# o200k_base — exact tokeniser for gpt-4o.
# Token counts are precise, not approximations.

print(f"Model    : {MODEL}")
print(f"Note     : No summary model — Token Buffer makes zero extra API calls")

Key loaded from environment variable.
Model    : gpt-4o
Note     : No summary model — Token Buffer makes zero extra API calls


---
## Part 1 — The Message Data Model

Same across all five techniques. `token_count()` is the key method here — it is called constantly during eviction decisions.

In [4]:
@dataclass
class Message:
    """A single conversation message — role, content, timestamp, token count."""

    role: str
    # 'user' or 'assistant'.

    content: str
    # The message text.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set at creation.

    def to_api_format(self) -> Dict[str, str]:
        """Return only role and content — what the OpenAI API accepts."""
        return {"role": self.role, "content": self.content}

    def token_count(self) -> int:
        """
        Count this message's tokens using the exact GPT-4o tokeniser.
        Called on every eviction check — must be fast.
        tiktoken.encode() is compiled C — it is fast enough for this.
        In production at very high throughput, consider caching this value
        on the object after first computation.
        """
        return len(TOKENISER.encode(self.content))

print("Message dataclass defined.")

Message dataclass defined.


---
## Part 2 — The TokenBufferMemory Class

The core class. The entire logic lives in two methods:
- `_evict_to_fit()` — drops oldest messages until the new message fits
- `add_message()` — calls `_evict_to_fit()` then appends

No summarisation. No external calls. Pure list manipulation.

In [5]:
class TokenBufferMemory:
    """
    A strictly token-budgeted message buffer.
    The buffer never exceeds max_buffer_tokens.
    When a new message would breach the limit, the oldest messages
    are dropped (evicted) until there is room — no compression, no LLM calls.

    This is the most operationally predictable of the five short-term
    memory techniques: zero extra API calls, zero latency spikes,
    exact cost per call = system_tokens + min(history, max_buffer_tokens).
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_buffer_tokens: int = 1_500,
        # Hard token ceiling for the message buffer.
        # No message can cause this to be exceeded.
        # Default 1,500: good for 8-12 typical FinCoach turns.
        # Unlike Technique 04, there is no 'summary zone' outside this budget —
        # this IS the entire context budget for conversation history.
        evict_in_pairs: bool = True,
        # When True: always evict complete turn pairs (user + assistant together).
        # Prevents orphaned messages (e.g. an assistant reply with no preceding question).
        # When False: evict one message at a time — maximises space efficiency
        # but may leave the context in a partial-turn state.
        # Production recommendation: always True unless you have a specific reason.
    ):
        self.session_id        = session_id
        self.user_id           = user_id
        self.system_prompt     = system_prompt
        self.max_buffer_tokens = max_buffer_tokens
        self.evict_in_pairs    = evict_in_pairs

        self._buffer: List[Message] = []
        # The active message buffer — token-budgeted, always verbatim.
        # Never exceeds max_buffer_tokens tokens total.
        # Grows on every add_message() and shrinks during eviction.

        self._evicted: List[Message] = []
        # All messages that have been evicted from the buffer.
        # In production: these should be written to a vector store or
        # database for retrieval (Technique 06). Here we keep them in
        # memory for inspection and stats.

        self._archive: List[Message] = []
        # The complete session history — every message, never evicted.
        # Not sent to the API. Available for compliance and retrieval.

        self._total_turns      = 0   # Complete turns ever (user+assistant pairs).
        self._total_evictions  = 0   # Total individual messages evicted.
        self._eviction_events  = 0   # Times eviction was triggered (one event may drop N msgs).
        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[TokenBuffer] Initialised — session: {self.session_id}")
        print(f"  Token budget  : {self.max_buffer_tokens:,} tokens")
        print(f"  Evict in pairs: {self.evict_in_pairs}")
        print(f"  Extra API calls on overflow: 0 (no compression — pure eviction)")

    # ------------------------------------------------------------------
    # TOKEN COUNTING
    # ------------------------------------------------------------------

    def _buffer_tokens(self) -> int:
        """Sum of tokens across all messages currently in the buffer."""
        return sum(m.token_count() for m in self._buffer)

    def _system_tokens(self) -> int:
        """Tokens in the system prompt — constant overhead per API call."""
        return len(TOKENISER.encode(self.system_prompt))

    def get_context_breakdown(self) -> Dict[str, int]:
        """Token breakdown for the next API call — by component."""
        sys = self._system_tokens()
        buf = self._buffer_tokens()
        return {
            "system": sys,
            "buffer": buf,
            "total":  sys + buf,
            # Note: no 'summary' zone — Token Buffer has no compressed state.
            # The cost model is simply: system + buffer, always.
        }

    # ------------------------------------------------------------------
    # EVICTION ENGINE
    # ------------------------------------------------------------------

    def _evict_to_fit(self, incoming_tokens: int) -> None:
        """
        Drop the oldest messages from the buffer until:
          buffer_tokens + incoming_tokens <= max_buffer_tokens

        This is the core mechanism that makes Token Buffer work.
        It is pure list manipulation — no API calls, no latency.

        Args:
            incoming_tokens: Token count of the message about to be added.
        """

        if not self._buffer:
            # Buffer is empty — nothing to evict.
            # This can happen if a single message is larger than the entire budget.
            # In production: log a warning and consider raising an error.
            return

        self._eviction_events += 1
        # Count this as one eviction event — may drop multiple messages.

        print(f"  [EVICT] Buffer at {self._buffer_tokens()} tokens, "
              f"incoming {incoming_tokens} tokens, "
              f"budget {self.max_buffer_tokens} — evicting...")

        evicted_in_this_event = 0
        # Track how many messages are dropped in this single eviction event.

        while self._buffer and \
              (self._buffer_tokens() + incoming_tokens) > self.max_buffer_tokens:
            # Keep looping while: buffer is non-empty AND still over budget.
            # Each loop iteration drops one message (or one pair if evict_in_pairs=True).

            if self.evict_in_pairs and len(self._buffer) >= 2:
                # Evict in pairs: drop the oldest TWO messages together.
                # This always removes a complete turn (user question + assistant answer).
                # Prevents the buffer from holding an orphaned response with no question.

                oldest_user      = self._buffer[0]  # The oldest user message.
                oldest_assistant = self._buffer[1]  # Its corresponding assistant response.

                # Verify the pair is actually user+assistant in order.
                # If not (e.g. session started mid-turn), fall back to single eviction.
                if oldest_user.role == "user" and oldest_assistant.role == "assistant":
                    self._evicted.append(oldest_user)
                    self._evicted.append(oldest_assistant)
                    # Move both to the evicted list before removing from buffer.
                    self._buffer = self._buffer[2:]
                    # Slice off the first two messages — O(n) but acceptable for
                    # typical buffer sizes (< 100 messages).
                    # For very high throughput: use collections.deque instead.
                    self._total_evictions += 2
                    evicted_in_this_event += 2
                    print(f"    Dropped pair: [{oldest_user.content[:35]}...] + response")
                else:
                    # Unexpected order — fall back to single message eviction.
                    evicted_msg = self._buffer.pop(0)
                    self._evicted.append(evicted_msg)
                    self._total_evictions += 1
                    evicted_in_this_event += 1
            else:
                # Single message eviction — when evict_in_pairs=False or buffer has 1 msg.
                evicted_msg = self._buffer.pop(0)
                # pop(0) removes and returns the FIRST (oldest) message.
                self._evicted.append(evicted_msg)
                # Archive the evicted message — in production, write to DB/vector store.
                self._total_evictions += 1
                evicted_in_this_event += 1
                print(f"    Dropped msg : [{evicted_msg.role}] '{evicted_msg.content[:45]}...'")

        print(f"  [EVICT] Done — dropped {evicted_in_this_event} messages, "
              f"buffer now {self._buffer_tokens()} tokens")

    # ------------------------------------------------------------------
    # CORE OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Add a new message to the buffer.
        If the budget would be exceeded, evict oldest messages first.
        This method always succeeds — unlike Technique 01 which raises BufferOverflowError.
        """

        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'. Use 'user' or 'assistant'.")

        new_message = Message(role=role, content=content)
        # Create the message — timestamp auto-set.

        incoming_tokens = new_message.token_count()
        # Count the new message's tokens BEFORE adding.

        if incoming_tokens > self.max_buffer_tokens:
            # Edge case: the new message ALONE exceeds the entire budget.
            # We cannot fit it no matter how much we evict.
            # Options: raise error, truncate the message, or increase the budget.
            # We raise here — the caller should handle this case explicitly.
            raise ValueError(
                f"Message too large: {incoming_tokens} tokens exceeds "
                f"the entire budget of {self.max_buffer_tokens} tokens. "
                f"Increase max_buffer_tokens or truncate the message."
            )

        projected = self._buffer_tokens() + incoming_tokens
        # What the buffer token count would be after adding the new message.

        if projected > self.max_buffer_tokens:
            # Budget would be exceeded — evict oldest messages first.
            self._evict_to_fit(incoming_tokens=incoming_tokens)
            # After this call, buffer_tokens + incoming_tokens <= max_buffer_tokens.

        self._buffer.append(new_message)
        # Append the new message — now guaranteed to fit within budget.

        self._archive.append(new_message)
        # Always archive — the complete session record, never evicted.

        if role == "assistant":
            self._total_turns += 1
            # Count completed turns — each assistant message = one complete turn.

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Build the complete message list for the OpenAI API.
        Structure: [system: persona] + [buffer messages verbatim]

        Note: unlike Techniques 03 and 04, there is NO summary block here.
        The API receives only the system prompt and the current buffer contents.
        This is the simplest possible context structure — and the most predictable.
        """

        messages = [{"role": "system", "content": self.system_prompt}]
        # System prompt first — always.

        for msg in self._buffer:
            messages.append(msg.to_api_format())
            # Add each buffer message in chronological order.
            # to_api_format() strips the timestamp — only role and content go to the API.

        return messages
        # Final structure: [system] + [recent verbatim messages within budget]
        # Total tokens: system_tokens + buffer_tokens <= system_tokens + max_buffer_tokens

    # ------------------------------------------------------------------
    # PERSISTENCE
    # ------------------------------------------------------------------

    def persist(self, filepath: str) -> None:
        """Save the complete session state to a JSON file."""
        ctx = self.get_context_breakdown()

        record = {
            "schema_version": "1.0",
            "technique": "token_buffer_memory",
            "session_id": self.session_id,
            "user_id": self.user_id,
            "model": MODEL,
            "created_at": self.created_at,
            "saved_at": datetime.now(timezone.utc).isoformat(),
            "config": {
                "max_buffer_tokens": self.max_buffer_tokens,
                "evict_in_pairs": self.evict_in_pairs,
            },
            "stats": {
                "total_turns": self._total_turns,
                "total_evictions": self._total_evictions,
                # Total individual messages dropped — key quality metric.
                # High evictions = budget too small or session too long.
                "eviction_events": self._eviction_events,
                # Times eviction was triggered — each event may drop multiple messages.
                "buffer_messages": len(self._buffer),
                "evicted_messages": len(self._evicted),
                "archive_messages": len(self._archive),
                "context_tokens": ctx,
            },
            "buffer": [asdict(m) for m in self._buffer],
            # Active buffer — what the agent currently sees.
            "evicted": [asdict(m) for m in self._evicted],
            # Evicted messages — in production, these go to a vector store.
            # Saved here for analysis and potential manual review.
            "archive": [asdict(m) for m in self._archive],
            # Full session history — every message, for compliance.
        }

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
            # ensure_ascii=False preserves ₹ and other non-ASCII characters.

        print(f"[TokenBuffer] Persisted — {self._total_turns} turns, "
              f"{self._total_evictions} messages evicted, "
              f"{ctx['total']} context tokens")

    @classmethod
    def load(cls, filepath: str) -> "TokenBufferMemory":
        """Restore TokenBufferMemory from a saved JSON file."""

        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)

        if record.get("schema_version") != "1.0":
            raise ValueError(f"Unknown schema version: {record.get('schema_version')}")

        mem = cls(
            session_id=record["session_id"],
            user_id=record["user_id"],
            system_prompt="[LOADED — set system_prompt after loading]",
            max_buffer_tokens=record["config"]["max_buffer_tokens"],
            evict_in_pairs=record["config"]["evict_in_pairs"],
        )

        def restore(msg_list):
            return [
                Message(role=m["role"], content=m["content"], timestamp=m["timestamp"])
                for m in msg_list
            ]

        mem._buffer  = restore(record["buffer"])
        mem._evicted = restore(record["evicted"])
        mem._archive = restore(record["archive"])
        mem._total_turns     = record["stats"]["total_turns"]
        mem._total_evictions = record["stats"]["total_evictions"]
        mem._eviction_events = record["stats"]["eviction_events"]
        mem.created_at       = record["created_at"]

        print(f"[TokenBuffer] Loaded — {mem._total_turns} turns, "
              f"{mem._total_evictions} evictions")
        return mem

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """Print a full breakdown of buffer state and eviction metrics."""
        ctx = self.get_context_breakdown()

        print(f"\n{'='*60}")
        print(f"  Token Buffer Stats — Session: {self.session_id}")
        print(f"{'='*60}")
        print(f"  Total turns ever      : {self._total_turns}")
        print(f"  Eviction events       : {self._eviction_events}")
        print(f"  Messages evicted      : {self._total_evictions}")
        print(f"  Messages in buffer    : {len(self._buffer)}")
        print(f"  Messages in archive   : {len(self._archive)}")
        print()
        print(f"  Token breakdown — next API call:")
        print(f"    System prompt : {ctx['system']:>6,} tokens  (constant)")
        print(f"    Buffer        : {ctx['buffer']:>6,} / {self.max_buffer_tokens:,} tokens")
        print(f"    ─────────────────────────────")
        print(f"    TOTAL input   : {ctx['total']:>6,} tokens")
        print(f"    No summary zone — cost model: system + buffer only")
        print()

        # Buffer utilisation bar.
        util = min(ctx['buffer'] / self.max_buffer_tokens, 1.0) if self.max_buffer_tokens > 0 else 0
        bar  = "█" * int(30 * util) + "░" * (30 - int(30 * util))
        print(f"  Buffer fill : [{bar}] {util:.0%} of {self.max_buffer_tokens:,} token budget")

        if self._evicted:
            print(f"\n  Evicted messages ({len(self._evicted)} total):")
            for msg in self._evicted[-4:]:
                # Show the last 4 evicted messages — most recent evictions.
                print(f"    [{msg.role:>9}] {msg.content[:55]}...")
            if len(self._evicted) > 4:
                print(f"    ... and {len(self._evicted) - 4} more")
        print(f"{'='*60}\n")


print("TokenBufferMemory class defined.")

TokenBufferMemory class defined.


---
## Part 3 — The FinCoach Agent

In [6]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- If you do not have information you need, explicitly ask for it rather than assuming.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions."""
# Note: "explicitly ask for it rather than assuming" — this surfaces the eviction
# problem clearly when early facts are dropped. The model will ask for the salary
# again rather than guessing, making the forgetting visible.


def chat(
    memory: TokenBufferMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """
    Send one user message to FinCoach. Return the assistant's reply.
    Eviction (if triggered) fires inside add_message() — transparent to the caller.
    No extra API calls, no latency overhead from memory management.
    """

    # STEP 1 — Add the user message.
    # If adding this message exceeds the budget, eviction fires automatically.
    # The buffer always accepts the message — no BufferOverflowError.
    memory.add_message(role="user", content=user_message)

    # STEP 2 — Call the API.
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
        # Returns: [system] + [buffer messages within budget]
        # No summary block — what you see is what the model gets.
    )

    # STEP 3 — Extract the reply.
    assistant_reply = response.choices[0].message.content

    # STEP 4 — Add the assistant reply.
    # Long responses can also trigger eviction — verbose answers consume budget.
    memory.add_message(role="assistant", content=assistant_reply)

    if verbose:
        ctx = memory.get_context_breakdown()
        print(f"[Turn {memory._total_turns}] "
              f"API input: {response.usage.prompt_tokens} tokens | "
              f"Buffer: {ctx['buffer']}/{memory.max_buffer_tokens} tokens | "
              f"Evicted: {memory._total_evictions} msgs total")

    return assistant_reply


print("FinCoach chat() function defined.")

FinCoach chat() function defined.


---
## Part 4 — Demo: Eviction in Action

We use a tight budget of 600 tokens to force eviction early — making the behaviour visible.

**Watch for:**
- `[EVICT]` lines showing exactly which messages are dropped and why
- The buffer token count staying bounded throughout
- Turn 6 asking for the salary — which was evicted in Turn 4 — and FinCoach asking again

In [7]:
tb_memory = TokenBufferMemory(
    session_id="session_tb_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_buffer_tokens=600,
    # Tight budget — forces eviction after 3-4 turns for demo visibility.
    # Production value: 1500-2500 for FinCoach advisory sessions.
    evict_in_pairs=True,
    # Always drop complete turn pairs to preserve context coherence.
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: salary — will be evicted once the budget fills.
    "My monthly expenses are ₹60,000 covering rent, food, and transport.",
    # Turn 2: expenses.
    "I have an FD of ₹50,000 maturing in 3 months. I'm conservative with money.",
    # Turn 3: FD and risk profile.
    "What is the difference between equity and debt mutual funds?",
    # Turn 4: generic question — budget pressure builds, Turn 1 may be evicted.
    "Which type is more suitable for someone risk-averse like me?",
    # Turn 5: uses risk profile from Turn 3.
    "Based on my exact salary, how much should I invest each month?",
    # Turn 6: requires salary from Turn 1 — which may have been evicted.
    # Expected: FinCoach asks for salary again OR answers incorrectly.
    # Either way it demonstrates the eviction failure clearly.
]

print("TOKEN BUFFER MEMORY DEMO — Eviction Visible")
print(f"Budget: 600 tokens | Evict in pairs: True")
print("=" * 65)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=tb_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
tb_memory.print_stats()

[TokenBuffer] Initialised — session: session_tb_demo_001
  Token budget  : 600 tokens
  Evict in pairs: True
  Extra API calls on overflow: 0 (no compression — pure eviction)
TOKEN BUFFER MEMORY DEMO — Eviction Visible
Budget: 600 tokens | Evict in pairs: True

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
[Turn 1] API input: 154 tokens | Buffer: 98/600 tokens | Evicted: 0 msgs total
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, it's great that you're thinking about managing your finances. To provide more tailored advice, could you share your current financial goals or areas you're interested in, such as savings, investments, or budgeting? Additionally, do you have any existing debts or financial commitments? This information will help me give you more specific guidance.

--- Turn 2 ---
User: My monthly expenses are ₹60,000 covering rent, food, and transport.
[Turn 2] API input: 257 tokens | Buffer: 310/600 tokens | Evicted: 0 msgs to

---
#### Part 5 — `evict_in_pairs=False` vs `evict_in_pairs=True`: Side-by-Side

In [8]:
def demonstrate_eviction_modes() -> None:
    """
    Show the difference between evicting individual messages vs paired messages.
    No API calls — simulates eviction using token counts only.
    """

    # Simulate a buffer at capacity with 3 complete turns.
    simulated_buffer = [
        Message(role="user",      content="My salary is ₹1,20,000 per month."),
        Message(role="assistant", content="Got it. What are your financial goals?"),
        Message(role="user",      content="My expenses are ₹60,000. I want to invest."),
        Message(role="assistant", content="With ₹60,000 surplus, you have good options."),
        Message(role="user",      content="What about SIPs for a conservative investor?"),
        Message(role="assistant", content="SIPs in debt funds suit your risk profile."),
    ]

    print("EVICTION MODE COMPARISON")
    print("=" * 65)
    print(f"Buffer has {len(simulated_buffer)} messages. Budget exceeded. Must drop messages.")
    print()

    # --- Mode 1: evict_in_pairs=True ---
    print("Mode 1: evict_in_pairs=True")
    print("Drop the oldest USER+ASSISTANT pair together.")
    after_pairs = simulated_buffer[2:]  # Drop first 2 (1 complete turn).
    print("Buffer after eviction:")
    for i, m in enumerate(after_pairs):
        print(f"  [{m.role:>9}] {m.content}")
    print(f"Result: 4 messages remain, all in valid user→assistant order.")
    print(f"Context coherence: ✅ preserved")
    print()

    # --- Mode 2: evict_in_pairs=False ---
    print("Mode 2: evict_in_pairs=False")
    print("Drop only the oldest single message.")
    after_single = simulated_buffer[1:]  # Drop only the first message (user).
    print("Buffer after eviction:")
    for i, m in enumerate(after_single):
        print(f"  [{m.role:>9}] {m.content}")
    print(f"Problem: buffer now starts with an ASSISTANT message with no preceding question.")
    print(f"The model sees a reply floating without context.")
    print(f"Context coherence: ❌ broken — orphaned assistant message at position [0]")
    print()

    print("Production recommendation: ALWAYS use evict_in_pairs=True.")
    print("The marginal token savings from single eviction are not worth the context incoherence.")


demonstrate_eviction_modes()

EVICTION MODE COMPARISON
Buffer has 6 messages. Budget exceeded. Must drop messages.

Mode 1: evict_in_pairs=True
Drop the oldest USER+ASSISTANT pair together.
Buffer after eviction:
  [     user] My expenses are ₹60,000. I want to invest.
  [assistant] With ₹60,000 surplus, you have good options.
  [     user] What about SIPs for a conservative investor?
  [assistant] SIPs in debt funds suit your risk profile.
Result: 4 messages remain, all in valid user→assistant order.
Context coherence: ✅ preserved

Mode 2: evict_in_pairs=False
Drop only the oldest single message.
Buffer after eviction:
  [assistant] Got it. What are your financial goals?
  [     user] My expenses are ₹60,000. I want to invest.
  [assistant] With ₹60,000 surplus, you have good options.
  [     user] What about SIPs for a conservative investor?
  [assistant] SIPs in debt funds suit your risk profile.
Problem: buffer now starts with an ASSISTANT message with no preceding question.
The model sees a reply floating with

---
## Part 6 — Cost Certainty: Why Token Buffer Has the Most Predictable Cost Model

In [9]:
def compare_cost_predictability(
    turns: List[str],
    max_budget: int = 1_500,
    avg_response_tokens: int = 100
) -> None:
    """
    Compare per-turn input token counts across all five short-term memory techniques.
    Shows why Token Buffer has the flattest, most predictable cost profile.
    No API calls — pure token arithmetic.
    """

    from collections import deque
    # deque for simulating the sliding window.

    sys_tokens = len(TOKENISER.encode(FINCOACH_SYSTEM_PROMPT))
    # System prompt — constant overhead for all techniques.

    # Simulate each technique in parallel.
    buf_history   = []          # Technique 01: grows forever.
    win_deque     = deque(maxlen=5 * 2)  # Technique 02: window of 5 turns.
    sum_history   = []          # Technique 03: summary + recent 3 turns.
    sb_buf_tokens = 0           # Technique 04: summary buffer.
    tb_buf_tokens = 0           # Technique 05: token buffer.

    SUMMARY_TOKEN_ESTIMATE = 120
    # Estimated tokens for the summary block in Techniques 03 and 04.
    # In practice this varies — we use a fixed estimate for comparison.

    print(f"System prompt: {sys_tokens} tokens (added to every technique)")
    print(f"Budget / window: {max_budget} tokens | Window: 5 turns | Avg response: {avg_response_tokens}t")
    print()
    print(f"{'Turn':>4} | {'01-Buffer':>10} | {'02-Window':>10} | "
          f"{'03-Summary':>10} | {'04-SumBuf':>10} | {'05-TokBuf':>10}")
    print("-" * 62)

    for i, user_msg in enumerate(turns, start=1):
        user_t = len(TOKENISER.encode(user_msg))
        resp_t = avg_response_tokens

        # --- 01: Buffer — grows forever ---
        buf_history.append(user_t)
        t01 = sys_tokens + sum(buf_history)
        buf_history.append(resp_t)  # Add response for next turn.

        # --- 02: Sliding Window — last 5 turns ---
        win_deque.append(user_t)
        t02 = sys_tokens + sum(win_deque)
        win_deque.append(resp_t)

        # --- 03: Summary Memory — summary + last 3 turns verbatim ---
        sum_history.append(user_t)
        keep_last = sum(sum_history[-6:])  # Last 3 turns = 6 messages.
        has_summary = len(sum_history) > 6
        t03 = sys_tokens + (SUMMARY_TOKEN_ESTIMATE if has_summary else 0) + keep_last
        sum_history.append(resp_t)

        # --- 04: Summary Buffer — summary + token-budgeted buffer ---
        sb_buf_tokens += user_t
        if sb_buf_tokens > max_budget:
            # Overflow: compress some messages (approximated as freeing 4*60 tokens).
            sb_buf_tokens = max(0, sb_buf_tokens - 4 * 60)
        has_summary_sb = sb_buf_tokens > max_budget * 0.3
        t04 = sys_tokens + (SUMMARY_TOKEN_ESTIMATE if has_summary_sb else 0) + min(sb_buf_tokens, max_budget)
        sb_buf_tokens += resp_t

        # --- 05: Token Buffer — hard cap, evict oldest ---
        tb_buf_tokens += user_t
        if tb_buf_tokens > max_budget:
            tb_buf_tokens = max_budget  # Eviction brings it back to the cap.
        t05 = sys_tokens + tb_buf_tokens
        tb_buf_tokens = min(tb_buf_tokens + resp_t, max_budget)  # Response also capped.

        print(f"{i:>4} | {t01:>10,} | {t02:>10,} | {t03:>10,} | {t04:>10,} | {t05:>10,}")

    print("-" * 62)
    print()
    print("Observations:")
    print("  01-Buffer  : grows every turn — highest cost at scale")
    print("  02-Window  : flat after window fills — but turn-count not token-precise")
    print("  03-Summary : flat but has summary overhead — varies with summary quality")
    print("  04-SumBuf  : flat + summary overhead — most complete memory")
    print("  05-TokBuf  : flattest and most predictable — hard ceiling, no summary overhead")


analysis_turns = [
    "Hi, my salary is ₹1,20,000 and expenses are ₹60,000.",
    "I'm 32, risk-averse, and want to retire at 55.",
    "I have an FD of ₹50,000 maturing next quarter.",
    "What is the difference between PPF and NPS?",
    "Which suits a conservative investor?",
    "How much should I invest monthly to reach ₹2 crore by 55?",
    "Should I invest my FD maturity amount in a lump sum SIP?",
    "What are the tax implications of NPS contributions?",
    "Should I open a PPF account given the 15-year lock-in?",
    "Give me a complete monthly investment breakdown.",
]

compare_cost_predictability(analysis_turns, max_budget=1_500)

System prompt: 124 tokens (added to every technique)
Budget / window: 1500 tokens | Window: 5 turns | Avg response: 100t

Turn |  01-Buffer |  02-Window | 03-Summary |  04-SumBuf |  05-TokBuf
--------------------------------------------------------------
   1 |        143 |        143 |        143 |        143 |        143
   2 |        259 |        259 |        259 |        259 |        259
   3 |        373 |        373 |        373 |        373 |        373
   4 |        484 |        484 |        585 |        484 |        484
   5 |        590 |        590 |        575 |        710 |        590
   6 |        705 |        686 |        576 |        825 |        705
   7 |        818 |        683 |        578 |        938 |        818
   8 |        928 |        679 |        582 |      1,048 |        928
   9 |      1,043 |        683 |        582 |      1,163 |      1,043
  10 |      1,151 |        685 |        577 |      1,271 |      1,151
---------------------------------------------

---
## Part 7 — Production Pattern: Token Buffer + Vector Store (Preview)

In [10]:
# This cell shows the production pattern in pseudocode — no real vector store needed.
# Technique 06 (Vector Store Memory) implements the full version.
# The point here: Token Buffer is most powerful when evicted messages are not lost.

print("PRODUCTION PATTERN: Token Buffer + Vector Store")
print("=" * 65)

print("""
┌─────────────────────────────────────────────────────────────┐
│                    Every conversation turn                   │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  TokenBufferMemory.add_message()                             │
│  ├── Check token budget                                      │
│  ├── If overflow: evict oldest messages                      │
│  │     └── WRITE evicted messages to vector store           │
│  │          (Technique 06 — ChromaDB / pgvector)             │
│  └── Append new message to buffer                           │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  Before each API call: retrieve relevant evicted messages   │
│  vector_store.search(query=current_user_message, k=3)       │
│  → Returns top-3 most semantically relevant evicted msgs    │
│  → Inject these as additional context before the buffer     │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  API call receives:                                          │
│  [system: persona]                                           │
│  [retrieved: top-3 relevant evicted msgs]  ← semantic match │
│  [buffer: recent N tokens verbatim]        ← recency        │
└─────────────────────────────────────────────────────────────┘

Result:
  ✅ Budget stays within max_buffer_tokens
  ✅ Evicted facts are retrievable by semantic similarity
  ✅ Salary from Turn 1 surfaces when the Turn 10 query is about salary
  ✅ Zero latency overhead — retrieval is a fast vector search
  ✅ No compression LLM calls
""")

print("This is the architecture we build in Technique 06 (Vector Store Memory).")
print("Token Buffer provides the active window. Vector Store provides the long-term layer.")

PRODUCTION PATTERN: Token Buffer + Vector Store

┌─────────────────────────────────────────────────────────────┐
│                    Every conversation turn                   │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  TokenBufferMemory.add_message()                             │
│  ├── Check token budget                                      │
│  ├── If overflow: evict oldest messages                      │
│  │     └── WRITE evicted messages to vector store           │
│  │          (Technique 06 — ChromaDB / pgvector)             │
│  └── Append new message to buffer                           │
└─────────────────────────────────────────────────────────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  Before each API call: retrieve relevant evicted messages   │
│  vector_store.search(query=current_user_message, k=3)       │
│  → R

---
## Part 8 — Session Persistence

In [11]:
SESSION_FILE = "/tmp/fincoach_tb_session_demo_001.json"

tb_memory.persist(SESSION_FILE)
# Saves buffer (active), evicted (dropped messages), and archive (full history).
# In production: evicted messages go to vector store, not just a JSON file.

print("\n--- Simulating service restart ---\n")

restored = TokenBufferMemory.load(SESSION_FILE)
restored.system_prompt = FINCOACH_SYSTEM_PROMPT

print("Continuing after restore:")
follow_up = "Can you remind me of the advice you gave me about my FD and what I should do with it?"
print(f"User: {follow_up}")

response = chat(memory=restored, user_message=follow_up, verbose=True)
# The restored buffer contains only what was in the buffer at save time —
# the recent messages within the token budget.
# Evicted messages from the demo are in restored._evicted but NOT in context.

print(f"FinCoach: {response}")

[TokenBuffer] Persisted — 6 turns, 4 messages evicted, 663 context tokens

--- Simulating service restart ---

[TokenBuffer] Initialised — session: session_tb_demo_001
  Token budget  : 600 tokens
  Evict in pairs: True
  Extra API calls on overflow: 0 (no compression — pure eviction)
[TokenBuffer] Loaded — 6 turns, 4 evictions
Continuing after restore:
User: Can you remind me of the advice you gave me about my FD and what I should do with it?
  [EVICT] Buffer at 560 tokens, incoming 137 tokens, budget 600 — evicting...
    Dropped pair: [I have an FD of ₹50,000 maturing in...] + response
  [EVICT] Done — dropped 2 messages, buffer now 368 tokens
[Turn 7] API input: 727 tokens | Buffer: 505/600 tokens | Evicted: 6 msgs total
FinCoach: Certainly! Since you are conservative with your money, once your ₹50,000 FD matures in 3 months, you might consider the following options:

1. **Reinvest in another FD**: This will continue to offer you stable returns with minimal risk.

2. **Consider Deb

---
## Key Takeaways

**What you built:** A `TokenBufferMemory` class with token-precise budget enforcement, pair-aware eviction, an eviction archive, session persistence, and a five-way cost comparison.

---

### The three things to remember

1. **Token Buffer is the simplest, most operationally predictable short-term memory.** No compression calls, no latency spikes, no summary quality concerns. Cost per call = system tokens + min(history, budget). Always. That formula is what makes it attractive for latency-sensitive and cost-audited production systems.

2. **Always evict in pairs.** Dropping individual messages breaks turn pairs and leaves orphaned assistant responses at the head of the buffer. This confuses the model in subtle ways. The fix — evict two messages at a time (one complete turn) — costs almost nothing extra and prevents context incoherence.

3. **Token Buffer + Vector Store is the production architecture.** On its own, Token Buffer loses evicted facts permanently. Paired with a vector store (Technique 06), evicted messages are indexed and retrievable by semantic similarity. The buffer handles recency. The vector store handles history. Together they cover everything without compression overhead.

---

### Crossing the session boundary — what comes next

All five techniques so far live entirely within a single session. When the session ends, the buffer resets — the user is a stranger again on their next visit.

 **Technique 06 — Vector Store Memory** is where this changes. 
 It introduces external persistent storage: past messages are embedded into vectors, stored in a database (ChromaDB, pgvector), and retrieved by semantic similarity at the start of every session. A returning user's salary, goals, and risk profile from three months ago surface automatically — not because they are in the buffer, but because the retrieval layer found them relevant to the current query.

That is the architectural shift from short-term to long-term memory. 

